# 📹 Smart Camera AI - Google Colab All-in-One Notebook

Real-time Computer Vision system powered by Google Colab Cloud GPU (NVIDIA T4):
- 🎯 **YOLO11 Object & Keypoint Detection**
- 💺 **Hierarchical Posture Classification** (Sitting vs. Standing)
- ⏱️ **Per-Person Dwell Timing & Transition Timestamps** ("Sat Down At", "Stood Up At")
- 📊 **Web Dashboard with Real-Time Video Streaming & Telemetry**

---
### ⚡ Quick Start:
1. Make sure your Colab Runtime has **GPU enabled** (`Runtime` ➔ `Change runtime type` ➔ `T4 GPU`).
2. Run the cells below in order.

### Step 1: Verify Colab GPU Acceleration

In [ ]:
!nvidia-smi
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active GPU    : {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ Warning: GPU is not active. Go to Runtime -> Change runtime type -> Select T4 GPU.")

### Step 2: Clone / Pull Repository from GitHub

In [ ]:
import os

REPO_URL = "https://github.com/devaakashchauhan/SmartCamera.git"
PROJECT_DIR = "/content/SmartCamera"

if not os.path.exists(PROJECT_DIR):
    print("Cloning repository from GitHub...")
    !git clone {REPO_URL} {PROJECT_DIR}
else:
    print("Repository already exists. Pulling latest updates...")
    %cd {PROJECT_DIR}
    !git pull

%cd {PROJECT_DIR}
print(f"Current Working Directory: {os.getcwd()}")

### Step 3: Install Required Dependencies & Cloudflared

In [ ]:
# 1. Install python packages
!pip install -q -r requirements.txt
!pip install -q uvicorn[standard] fastapi pydantic sqlalchemy websockets python-multipart

# 2. Install Cloudflared binary for secure public HTTPS tunneling
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1

print("✅ Environment configured successfully!")

### Step 4: Launch FastAPI Backend & Cloudflare Public Tunnel

This cell starts the Smart Camera backend server on Colab's GPU and creates a secure public tunnel link.
When the link appears, **click it on your laptop** to open the live dashboard!

In [ ]:
import subprocess
import time
import requests
import re

# 1. Stop any previously running server processes
!pkill -f uvicorn > /dev/null 2>&1
!pkill -f cloudflared > /dev/null 2>&1
time.sleep(1)

# 2. Start FastAPI backend with Colab GPU in background
print("🚀 Starting Smart Camera FastAPI backend server...")
server_log = open("server.log", "w")
server_process = subprocess.Popen(
    ["python", "-m", "uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000", "--app-dir", "backend"],
    stdout=server_log,
    stderr=subprocess.STDOUT
)

# 3. Verify backend health
time.sleep(3)
try:
    res = requests.get("http://127.0.0.1:8000/health", timeout=3)
    if res.status_code == 200:
        print("✅ Backend server is online and healthy:", res.json())
except Exception as e:
    print("ℹ️ Backend starting up...")

# 4. Launch Cloudflare Tunnel and display public URL
print("🌐 Creating secure public tunnel...")
tunnel_process = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

public_url = None
start_time = time.time()

for line in tunnel_process.stdout:
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        public_url = match.group(0)
        break
    if time.time() - start_time > 30:
        break

if public_url:
    print("=" * 65)
    print("🎉 SMART CAMERA AI DASHBOARD IS READY!")
    print(f"👉 Open Link: {public_url}")
    print("=" * 65)
    print("\nHow to stream live video:")
    print("  1. Click the link above on your laptop.")
    print("  2. Click the 'Share Laptop Webcam' button under the player.")
    print("  3. Allow webcam permission in your browser.")
    print("  4. Watch real-time AI bounding boxes, sit/stand stats & dwell times!")
else:
    print("⚠️ Could not automatically parse tunnel URL. Check tunnel logs.")

### Step 5: Live Server Logs & Diagnostics
Run this cell anytime while the system is running to see AI inferences and server events.

In [ ]:
!tail -n 40 server.log

### Step 6: Stop Server
Run this cell whenever you want to shut down the backend and tunnel.

In [ ]:
!pkill -f uvicorn
!pkill -f cloudflared
print("🛑 Smart Camera backend and Cloudflare tunnel stopped.")